In [58]:
import pandas as pd
from pathlib import Path

In [69]:
# ----------------------
# CONFIGURATION SECTION
# ----------------------

# --- Years ---
CURRENT_YEAR = 2026
PRIOR_YEAR = 2025

# --- Mode toggle ---
# False = build levy list fresh from current secured roll
# True  = seed levy list from prior-year final levy (analysis workbook style)
USE_PRIOR_YEAR_LEVY = False

# --- Parcel tax constants ---
PARCEL_TAX_AMOUNT = 59.00
TAX_CODE = "64504"

# --- Mammoth USD TRAs (from Board resolution Exhibit A) ---
MAMMOTH_TRAS = [
    "010-000", "010-001", "010-002", "010-003", "010-004", "010-005", "010-006", "010-007",
    "010-008", "010-009", "010-010", "010-011", "010-012", "010-013", "010-014", "010-015",
    "059-000", "059-005", "059-007", "059-012", "059-018",
]

# --- File locations (TEMP: single analysis workbook) ---
ANALYSIS_WORKBOOK_NAME = "601_Secured_2025 - Analysis.xlsx"
ANALYSIS_WORKBOOK_PATH = Path.home() / "Desktop" / ANALYSIS_WORKBOOK_NAME

# --- Sheet names inside analysis workbook ---
SECURED_CURRENT_SHEET = "AGENCYCDCURRSEC_TR601"
NONTAXABLE_CURRENT_SHEET = "2025 Nontaxable"
SENIOR_PRIOR_SHEET = "Prior Year Senior Exemptions"
PRIOR_FINAL_LEVY_SHEET = "FINAL 2025-26 Tax Levy File"   # used only if USE_PRIOR_YEAR_LEVY = True


### Load in File

In [70]:
from pathlib import Path
import pandas as pd

# =========================
# FILE IMPORT SECTION
# =========================

print("Using analysis workbook:")
print(ANALYSIS_WORKBOOK_PATH)
print("Workbook exists:", ANALYSIS_WORKBOOK_PATH.exists())

# List available sheets
xls = pd.ExcelFile(ANALYSIS_WORKBOOK_PATH)
print("\nSheets in workbook:")
for s in xls.sheet_names:
    print(" -", s)

# --- Load current-year secured roll ---
secured_df = pd.read_excel(
    ANALYSIS_WORKBOOK_PATH,
    sheet_name=SECURED_CURRENT_SHEET
)

print("\nLoaded secured roll:")
print("  rows:", len(secured_df))
print("  cols:", len(secured_df.columns))

# --- Load current-year non-taxable APNs ---
nontaxable_df = pd.read_excel(
    ANALYSIS_WORKBOOK_PATH,
    sheet_name=NONTAXABLE_CURRENT_SHEET
)

print("\nLoaded 2025 non-taxable list:")
print("  rows:", len(nontaxable_df))
print("  cols:", len(nontaxable_df.columns))

# --- Load prior-year senior exemptions ---
senior_df = pd.read_excel(
    ANALYSIS_WORKBOOK_PATH,
    sheet_name=SENIOR_PRIOR_SHEET
)

print("\nLoaded prior-year senior exemptions:")
print("  rows:", len(senior_df))
print("  cols:", len(senior_df.columns))

# --- Load prior-year final levy (ONLY if toggle enabled) ---
prior_levy_df = None
if USE_PRIOR_YEAR_LEVY:
    prior_levy_df = pd.read_excel(
        ANALYSIS_WORKBOOK_PATH,
        sheet_name=PRIOR_FINAL_LEVY_SHEET,
        header=None
    )
    print("\nLoaded prior-year final levy:")
    print("  rows:", len(prior_levy_df))
    print("  cols:", len(prior_levy_df.columns))

# Preview (optional)
secured_df.head(5)


Using analysis workbook:
/Users/ewilson/Desktop/601_Secured_2025 - Analysis.xlsx
Workbook exists: True

Sheets in workbook:
 - Instructions
 - AGENCYCDCURRSEC_TR601
 - 2024-25 Tax Levy File
 - PREVIOUS Nontaxable APNs
 - 2025 Nontaxable
 - Prior Year Senior Exemptions
 - New Senior Exemption
 - VSE that Changed
 - 2025 VSE
 - MammothTRAs2025
 - NEW APNs 2025
 - FINAL 2025-26 Tax Levy File

Loaded secured roll:
  rows: 17267
  cols: 44

Loaded 2025 non-taxable list:
  rows: 45
  cols: 25

Loaded prior-year senior exemptions:
  rows: 54
  cols: 16


,Tax Year,APN,Fee Parcel,TRA,Taxability,Owner,Assessee,MailAddress1,MailAddress2,MailAddress3,...,Total Rooms,Garage,Garage Size,Heating,AC,Fire Place,Pool Spa,Stories,Units,Current Document Number
0,2025,1020002000,1020002000,51001,0,WALKER RIVER IRRIGATION DIST.,WALKER RIVER IRRIGATION DIST.,P.O. BOX 820,YERINGTON NV 89447,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2008ICONV
1,2025,1020005000,1020005000,51001,0,WALKER RIVER IRRIGATION DIST.,WALKER RIVER IRRIGATION DIST.,P.O. BOX 820,YERINGTON NV 89447,NaN,...,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.0,2008ICONV
2,2025,1020006000,1020006000,51001,0,WALKER RIVER IRRIGATION DIST.,WALKER RIVER IRRIGATION DIST.,P.O. BOX 820,YERINGTON NV 89447,NaN,...,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.0,2008ICONV
3,2025,1020007000,1020007000,51001,0,WALKER RIVER IRRIGATION DIST.,WALKER RIVER IRRIGATION DIST.,P.O. BOX 820,YERINGTON NV 89447,NaN,...,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.0,2008ICONV
4,2025,1020008000,1020008000,51001,0,WALKER RIVER IRRIGATION DIST.,WALKER RIVER IRRIGATION DIST.,P.O. BOX 820,YERINGTON NV 89447,NaN,...,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.0,2008ICONV


### Verify TRAs (Work In Progress)

In [60]:
# =========================
# SETUP STEP 2: Identify key columns (APN, TRA, Owner)
# =========================

print("Total columns:", len(SECURED_COLS))
print("\nAll column names:")
for c in SECURED_COLS:
    print(" -", c)

# These are our expected names (edit if your file uses different headers)
APN_COL = "APN"
TRA_COL = "TRA"
OWNER_COL = "Owner"

missing = [c for c in [APN_COL, TRA_COL, OWNER_COL] if c not in SECURED_DF.columns]
if missing:
    raise KeyError(
        f"Missing expected column(s): {missing}\n"
        "Update APN_COL / TRA_COL / OWNER_COL to match your actual headers above."
    )

print("\nKey columns confirmed:")
print("APN_COL  =", APN_COL)
print("TRA_COL  =", TRA_COL)
print("OWNER_COL=", OWNER_COL)

print("\nSample APNs:")
print(SECURED_DF[APN_COL].dropna().head(10).tolist())

print("\nSample TRAs:")
print(SECURED_DF[TRA_COL].dropna().head(10).tolist())

print("\nSample Owners:")
print(SECURED_DF[OWNER_COL].dropna().head(10).tolist())


Total columns: 44

All column names:
 - Tax Year
 - APN
 - Fee Parcel
 - TRA
 - Taxability
 - Owner
 - Assessee
 - MailAddress1
 - MailAddress2
 - MailAddress3
 - MailAddress4
 - Situs1
 - Situs2
 - Parcel Description
 - Land Value
 - Structure Value
 - Fixtures Value
 - Growing Value
 - Fixtures RP Value
 - MHPP Value
 - Personal Property Value
 - HOX
 - Other Exemption
 - Other Exemption Code
 - Land Size Ft2
 - Acres
 - Building Type
 - Quality Class
 - Building Size
 - Year Built
 - Total Area
 - Bedrooms
 - Baths
 - Half Baths
 - Total Rooms
 - Garage
 - Garage Size
 - Heating
 - AC
 - Fire Place
 - Pool Spa
 - Stories
 - Units
 - Current Document Number

Key columns confirmed:
APN_COL  = APN
TRA_COL  = TRA
OWNER_COL= Owner

Sample APNs:
[1020002000, 1020005000, 1020006000, 1020007000, 1020008000, 1020009000, 1020011000, 1020013000, 1020014000, 1020016000]

Sample TRAs:
[51001, 51001, 51001, 51001, 51001, 51001, 51001, 51001, 51001, 51001]

Sample Owners:
['WALKER RIVER IRRIGATION

### Filter to Mammoth TRAs to produce the "Universe"

In [71]:
# =========================
# STAGE 1: Mammoth parcels (fix TRA format, in-memory only)
# =========================

APN_COL = "APN"
TRA_COL = "TRA"

# Convert TRA like 10006 -> "010-006", 59007 -> "059-007"
tra6 = secured_df[TRA_COL].astype("Int64").astype(str).str.zfill(6)
secured_df["TRA_norm"] = tra6.str[:3] + "-" + tra6.str[3:]

mammoth_df = secured_df[secured_df["TRA_norm"].isin(MAMMOTH_TRAS)].copy()

print("Mammoth parcel rows:", len(mammoth_df))
print("\nMammoth TRA counts:")
print(mammoth_df["TRA_norm"].value_counts())

mammoth_df.head(5)


Mammoth parcel rows: 11070

Mammoth TRA counts:
TRA_norm
010-006    7446
010-005    1158
010-008     559
010-001     524
059-007     518
010-004     316
059-012     217
010-009     132
059-005      94
059-000      57
010-007      29
010-000       9
010-002       9
010-014       1
010-015       1
Name: count, dtype: int64


,Tax Year,APN,Fee Parcel,TRA,Taxability,Owner,Assessee,MailAddress1,MailAddress2,MailAddress3,...,Garage,Garage Size,Heating,AC,Fire Place,Pool Spa,Stories,Units,Current Document Number,TRA_norm
2123,2025,14220005000,14220005000,59000,0,"ALPERS RANCH, LC","ALPERS RANCH, LC",JOHN GOTTWALD,"1100 BOULDERS PARKWAY, SUITE 101",RICHMOND VA 23225,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.0,2008R000243,059-000
2124,2025,14220006000,14220006000,59000,0,"ALPERS RANCH, LC","ALPERS RANCH, LC",JOHN GOTTWALD,"1100 BOULDERS PARKWAY, SUITE 101",RICHMOND VA 23225,...,NaN,NaN,5.0,NaN,NaN,NaN,1.0,1.0,2008R000243,059-000
2125,2025,14220014000,14220014000,59000,0,"ALPERS RANCH, LC","ALPERS RANCH, LC",JOHN GOTTWALD,"1100 BOULDERS PARKWAY, SUITE 101",RICHMOND VA 23225,...,NaN,NaN,1.0,NaN,1.0,NaN,2.0,1.0,2008R000243,059-000
2126,2025,14220015000,14220015000,59000,0,"ALPERS RANCH, LC","ALPERS RANCH, LC",JOHN GOTTWALD,"1100 BOULDERS PARKWAY, SUITE 101",RICHMOND VA 23225,...,NaN,NaN,NaN,NaN,NaN,N,NaN,NaN,2008R000243,059-000
2127,2025,14220016000,14220016000,59000,0,ARCULARIUS RANCH LC-TREDEGAR,ARCULARIUS RANCH LC-TREDEGAR,JOHN GOTTWALD,"1100 BOULDERS PARKWAY, SUITE 101",RICHMOND VA 23225,...,NaN,NaN,NaN,NaN,NaN,N,NaN,NaN,1998R000195,059-000


In [72]:
# =========================
# STAGE 2A: Apply CURRENT YEAR non-taxable exclusions
# Purpose:
#   Remove parcels that are permanently non-taxable from the levy universe.
#   These APNs should NEVER appear in the final levy file.
# =========================

base_df = mammoth_df.copy()   # <-- later this will respect your toggle

# --- Ensure APN_12 exists on base universe ---
if "APN_12" not in base_df.columns:
    base_df["APN_12"] = (
        base_df["APN"].astype(str)
        .str.replace(r"\D", "", regex=True)
        .str.zfill(12)
    )

# --- Identify APN column in non-taxable sheet ---
possible_apn_cols = [
    "APN_12", "APN", "APN NUMBER", "APN Number",
    "NUMBER VALUE", "Number Value", "FeeParcel", "Fee Parcel"
]

NONTAX_APN_COL = next(
    (c for c in possible_apn_cols if c in nontaxable_df.columns),
    None
)

if NONTAX_APN_COL is None:
    raise ValueError(
        "Could not identify APN column in 2025 Nontaxable sheet. "
        f"Columns found: {list(nontaxable_df.columns)}"
    )

# --- Normalize non-taxable APNs ---
nontaxable_df["APN_12"] = (
    nontaxable_df[NONTAX_APN_COL].astype(str)
    .str.replace(r"\D", "", regex=True)
    .str.zfill(12)
)

# --- Build comparison sets ---
nontax_set = set(nontaxable_df["APN_12"].dropna())
base_set = set(base_df["APN_12"].dropna())

matched_nontax = sorted(nontax_set & base_set)
unmatched_nontax = sorted(nontax_set - base_set)

# --- Apply exclusion ---
before_count = len(base_df)
after_nontax_df = base_df[~base_df["APN_12"].isin(matched_nontax)].copy()
removed_count = before_count - len(after_nontax_df)

# --- Diagnostics ---
print("\n=== STAGE 2A: NON-TAXABLE EXCLUSIONS ===")
print("APN column used:", NONTAX_APN_COL)
print("Non-taxable APNs (unique):", len(nontax_set))
print("Matched to levy universe:", len(matched_nontax))
print("Unmatched (not in universe):", len(unmatched_nontax))
print("Parcels before:", before_count)
print("Removed (non-taxable):", removed_count)
print("Remaining:", len(after_nontax_df))

# Optional review list
unmatched_nontax[:25]



=== STAGE 2A: NON-TAXABLE EXCLUSIONS ===
APN column used: APN
Non-taxable APNs (unique): 45
Matched to levy universe: 45
Unmatched (not in universe): 0
Parcels before: 11070
Removed (non-taxable): 45
Remaining: 11025


[]

In [82]:
# =========================
# STAGE 2B (REFINED): Senior owner comparison with stronger normalization
# Goal: reduce false "needs review" by treating punctuation / spacing / common words as non-differences
# - If Senior "Current Owner" is blank, fall back to "Master File Owner"
# - Build a "MatchScore" using token overlap + string similarity
# - Only flag Needs_Review when similarity is below thresholds
# =========================

from difflib import SequenceMatcher

# -------------------------
# Base universe for review (toggle-aware)
# -------------------------
# If USE_PRIOR_YEAR_LEVY is True:
#   we review senior APNs against the levy-base universe (prior list after non-taxable exclusions).
# If False:
#   we review senior APNs against the new-list universe (current Mammoth parcels after non-taxable).
#
# NOTE: 2B is review-only either way. It should not remove anything.
if "USE_PRIOR_YEAR_LEVY" in globals() and USE_PRIOR_YEAR_LEVY:
    base_df = prior_levy_df.copy()
else:
    base_df = after_nontax_df.copy()

# Ensure base_df has APN_12
if "APN_12" not in base_df.columns:
    base_df["APN_12"] = (
        base_df["APN"].astype(str)
        .str.replace(r"\D", "", regex=True)
        .str.zfill(12)
    )

# Ensure TRA_norm exists on base_df (for display)
if "TRA_norm" not in base_df.columns and "TRA" in base_df.columns:
    tra6 = base_df["TRA"].astype("Int64").astype(str).str.zfill(6)
    base_df["TRA_norm"] = tra6.str[:3] + "-" + tra6.str[3:]

# -------------------------
# Build senior_review_df (join senior list to current base universe Owner/TRA)
# -------------------------
senior_review_df = senior_df.copy()

# Normalize senior APN_12
if "APN_12" not in senior_review_df.columns:
    if "APN" not in senior_review_df.columns:
        raise ValueError("Senior sheet missing APN column (expected 'APN').")
    senior_review_df["APN_12"] = (
        senior_review_df["APN"].astype(str)
        .str.replace(r"\D", "", regex=True)
        .str.zfill(12)
    )

SECURED_OWNER_COL = "Owner"  # from secured roll join

# Pull Owner + TRA_norm from the base universe
lookup_cols = ["APN_12"]
if SECURED_OWNER_COL in base_df.columns:
    lookup_cols.append(SECURED_OWNER_COL)
if "TRA_norm" in base_df.columns:
    lookup_cols.append("TRA_norm")

base_lookup = base_df[lookup_cols].drop_duplicates(subset=["APN_12"])

# Join so senior_review_df now has the secured roll Owner (and TRA_norm if available)
senior_review_df = senior_review_df.merge(base_lookup, on="APN_12", how="left")


# --- Helper: strong name normalization (still conservative) ---
COMMON_DROP_TOKENS = {
    # very common legal/owner filler
    "TRUST", "TRUSTEE", "TRUSTEES", "FAMILY", "REVOCABLE", "LIVING",
    "ET", "AL", "ETAL", "THE",
    # joiners
    "&", "AND",
}

def _clean_owner_text(s: str) -> str:
    if s is None:
        return ""
    s = str(s)
    if s.lower() == "nan":
        return ""
    s = s.upper().strip()

    # Replace punctuation with spaces
    for ch in [",", ".", ":", ";", "(", ")", "[", "]", "{", "}", "'", '"', "-", "_", "/", "\\"]:
        s = s.replace(ch, " ")

    # Collapse multiple spaces
    s = " ".join(s.split())
    return s

def _owner_tokens(s: str) -> list:
    s = _clean_owner_text(s)
    if not s:
        return []
    toks = s.split()
    # drop very common filler tokens
    toks = [t for t in toks if t not in COMMON_DROP_TOKENS]
    return toks

def _jaccard(a: set, b: set) -> float:
    if not a and not b:
        return 1.0
    if not a or not b:
        return 0.0
    return len(a & b) / len(a | b)

def _seq_ratio(a: str, b: str) -> float:
    a = _clean_owner_text(a)
    b = _clean_owner_text(b)
    if not a and not b:
        return 1.0
    if not a or not b:
        return 0.0
    return SequenceMatcher(None, a, b).ratio()

def _match_score(name_a: str, name_b: str) -> float:
    # Token overlap captures punctuation/spacing differences well
    ta = set(_owner_tokens(name_a))
    tb = set(_owner_tokens(name_b))
    jac = _jaccard(ta, tb)

    # Sequence ratio helps with minor typos, date formatting, etc.
    seq = _seq_ratio(name_a, name_b)

    # Weighted blend (token overlap slightly more important)
    return 0.6 * jac + 0.4 * seq


# --- Build the "effective senior owner" (Current Owner if present else Master Owner) ---
# senior_review_df is produced in the prior Stage 2B cell (join already done)
# Columns we saw exist: "Master File Owner", "Current Owner" (senior sheet), and "Owner" (secured roll)

SENIOR_CURR_COL = "Current Owner" if "Current Owner" in senior_review_df.columns else None
SENIOR_MASTER_COL = "Master File Owner" if "Master File Owner" in senior_review_df.columns else None
SECURED_OWNER_COL = "Owner"  # from secured roll join

def _pick_effective_owner(row) -> str:
    curr = row.get(SENIOR_CURR_COL) if SENIOR_CURR_COL else None
    if curr is not None and str(curr).strip() != "" and str(curr).lower() != "nan":
        return curr
    mast = row.get(SENIOR_MASTER_COL) if SENIOR_MASTER_COL else None
    return mast

senior_review_df["Senior_Effective_Owner"] = senior_review_df.apply(_pick_effective_owner, axis=1)

# --- Compute similarity score ---
senior_review_df["MatchScore"] = senior_review_df.apply(
    lambda r: _match_score(r.get("Senior_Effective_Owner", ""), r.get(SECURED_OWNER_COL, "")),
    axis=1
)

# --- Decide review flag ---
# Thresholds chosen to eliminate obvious punctuation/casing differences while still catching true sales.
# If you want fewer flagged, drop threshold slightly (e.g., 0.88).
REVIEW_THRESHOLD = 0.90

senior_review_df["Needs_Review"] = senior_review_df["MatchScore"] < REVIEW_THRESHOLD

print("Senior owner compare using:")
print("  Base universe toggle :", "PRIOR YEAR LEVY BASE" if ("USE_PRIOR_YEAR_LEVY" in globals() and USE_PRIOR_YEAR_LEVY) else "NEW LIST BASE")
print("  Senior current col:", SENIOR_CURR_COL)
print("  Senior master col :", SENIOR_MASTER_COL)
print("  Secured owner col :", SECURED_OWNER_COL)
print("  Review threshold  :", REVIEW_THRESHOLD)

print("\nNeeds review count:", int(senior_review_df["Needs_Review"].sum()))
print("Top 10 lowest match scores:")
display(
    senior_review_df.sort_values("MatchScore")
    .loc[:, ["APN_12", "Add to Roll? (Y/N)", "Senior_Effective_Owner", SECURED_OWNER_COL, "TRA_norm", "MatchScore", "Needs_Review"]]
    .head(10)
)

print("\nSample of rows flagged for review (first 25):")
display(
    senior_review_df.loc[senior_review_df["Needs_Review"]]
    .loc[:, ["APN_12", "Add to Roll? (Y/N)", "Senior_Effective_Owner", SECURED_OWNER_COL, "TRA_norm", "MatchScore"]]
    .head(25)
)

print("\nSample of rows NOT flagged (first 10):")
display(
    senior_review_df.loc[~senior_review_df["Needs_Review"]]
    .loc[:, ["APN_12", "Senior_Effective_Owner", SECURED_OWNER_COL, "MatchScore"]]
    .head(10)
)

# Keep pipeline output unchanged (review-only stage)
after_senior_df = base_df.copy()


Senior owner compare using:
  Base universe toggle : NEW LIST BASE
  Senior current col: Current Owner
  Senior master col : Master File Owner
  Secured owner col : Owner
  Review threshold  : 0.9

Needs review count: 10
Top 10 lowest match scores:


,APN_12,Add to Roll? (Y/N),Senior_Effective_Owner,Owner,TRA_norm,MatchScore,Needs_Review
27,060170023000,N,KNOTT KATHRYN,FARIS LIVING TRUST 8-19-24,059-007,0.082051,True
25,040021007000,N,JASTRAB DOUG,DEL FANTE JAMES M. & KIM CHRISTINA,010-006,0.088889,True
47,039060011000,N,WAHL DAVID C & MARGARET A,MURRAY TRUST 9-30-10,010-006,0.088889,True
42,035270009000,N,SAVAGE FAMILY TRUST 02-14-18,JIANG ATHENA Y. & KRAFT BENJAMIN,010-006,0.108475,True
13,062090011000,N,FORSTENZER FAMILY TRUST,AGEE REVOCABLE TRUST 4-28-99,059-012,0.125490,True
31,060170014000,N,LUDVIK WILLIAM A & EILEEN,LUDVIK REVOCABLE TRUST 2-7-24,059-007,0.233862,True
44,039070010000,N,SHUGART LEE A & ANNA E,SHUGART LIVING TUST 4-9-92,010-006,0.283333,True
16,062120014000,N,GILBREATH FAMILY TRUST,GILBREATH FAMILY TRUST 12-17-82,059-005,0.482075,True
23,062130001000,N,HARZARD FAMILY TRUST 08-17-16,HAZARD FAMILY TRUST 8-17-16,059-005,0.585714,True
48,032130013000,N,WALTERS FAMILY TRUST 02-15-01,WALTERS FAMILY TRUST 2-15-01,010-001,0.752982,True



Sample of rows flagged for review (first 25):


,APN_12,Add to Roll? (Y/N),Senior_Effective_Owner,Owner,TRA_norm,MatchScore
13,062090011000,N,FORSTENZER FAMILY TRUST,AGEE REVOCABLE TRUST 4-28-99,059-012,0.125490
16,062120014000,N,GILBREATH FAMILY TRUST,GILBREATH FAMILY TRUST 12-17-82,059-005,0.482075
23,062130001000,N,HARZARD FAMILY TRUST 08-17-16,HAZARD FAMILY TRUST 8-17-16,059-005,0.585714
25,040021007000,N,JASTRAB DOUG,DEL FANTE JAMES M. & KIM CHRISTINA,010-006,0.088889
27,060170023000,N,KNOTT KATHRYN,FARIS LIVING TRUST 8-19-24,059-007,0.082051
31,060170014000,N,LUDVIK WILLIAM A & EILEEN,LUDVIK REVOCABLE TRUST 2-7-24,059-007,0.233862
42,035270009000,N,SAVAGE FAMILY TRUST 02-14-18,JIANG ATHENA Y. & KRAFT BENJAMIN,010-006,0.108475
44,039070010000,N,SHUGART LEE A & ANNA E,SHUGART LIVING TUST 4-9-92,010-006,0.283333
47,039060011000,N,WAHL DAVID C & MARGARET A,MURRAY TRUST 9-30-10,010-006,0.088889
48,032130013000,N,WALTERS FAMILY TRUST 02-15-01,WALTERS FAMILY TRUST 2-15-01,010-001,0.752982



Sample of rows NOT flagged (first 10):


,APN_12,Senior_Effective_Owner,Owner,MatchScore
0,033290037000,BAGGETT L RICHARD,"BAGGETT, L. RICHARD",1.0
1,033134014000,BENES FAMILY TRUST DTD 022007,BENES FAMILY TRUST DTD 022007,1.0
2,035267009000,BERMAN REVOCABLE TRUST,BERMAN REVOCABLE TRUST,1.0
3,039020027000,HUGHES LIVING TRUST 4-22-93,HUGHES LIVING TRUST 4-22-93,1.0
4,040021021000,BRUNS LESLIE,BRUNS LESLIE,1.0
5,035212014000,COLLINS FAMILY TRUST 11-30-03 ET AL & CHAIR 14...,COLLINS FAMILY TRUST 11-30-03 ET AL & CHAIR 14...,1.0
6,032130012000,BRAD & JAN DAIGLE TRUST,BRAD & JAN DAIGLE TRUST,1.0
7,033162214000,CRUZAN TRUST 11/03/2010,CRUZAN TRUST 11/03/2010,1.0
8,060160017000,CZESCHIN LIVING TRUST 01-06-05,CZESCHIN LIVING TRUST 01-06-05,1.0
9,035270001000,"DAMATO, VERILEE T.","DAMATO, VERILEE T.",1.0


In [83]:
# =========================
# STAGE 3: Build final levy table (in-memory)
# Purpose:
#   Produce the final parcel tax levy list for county submission.
#   Each APN appears once with a fixed parcel tax amount and tax code.
# =========================

final_df = after_senior_df.copy()

# --- Normalize APN to 12-digit string (required county format) ---
final_df["APN"] = (
    final_df["APN"]
    .astype(str)
    .str.replace(r"\D", "", regex=True)
    .str.zfill(12)
)

# --- Apply parcel tax constants (from Board resolution) ---
final_df["Fee"] = PARCEL_TAX_AMOUNT      # e.g. 59.00
final_df["TaxCode"] = TAX_CODE           # e.g. "64504"

# --- Build final levy output ---
levy_df = (
    final_df[["APN", "Fee", "TaxCode"]]
    .drop_duplicates(subset=["APN"])
    .sort_values("APN")
    .reset_index(drop=True)
)

print("Final levy rows:", len(levy_df))
levy_df.head(20)


Final levy rows: 11025


,APN,Fee,TaxCode
0,014220005000,59.0,64504
1,014220006000,59.0,64504
2,014220014000,59.0,64504
3,014220015000,59.0,64504
4,014220016000,59.0,64504
5,014220017000,59.0,64504
6,014220018000,59.0,64504
7,014220019000,59.0,64504
8,014220020000,59.0,64504
9,014240008000,59.0,64504


### QA

In [65]:
analysis_path = Path.home() / "Desktop" / "601_Secured_2025 - Analysis.xlsx"

FINAL_SHEET_NAME = "FINAL 2025-26 Tax Levy File"

final_analysis_df = pd.read_excel(
    analysis_path,
    sheet_name=FINAL_SHEET_NAME,
    header=None
)

print("Loaded analysis final levy sheet")
print("Rows:", len(final_analysis_df))
final_analysis_df.head(10)


Loaded analysis final levy sheet
Rows: 11000


,0,1,2
0,2.237001e+10,59.0,64504.0
1,3.904000e+10,59.0,64504.0
2,4.004241e+10,59.0,64504.0
3,6.021001e+10,59.0,64504.0
4,1.422000e+10,59.0,64504.0
5,1.422001e+10,59.0,64504.0
6,1.422001e+10,59.0,64504.0
7,1.422002e+10,59.0,64504.0
8,1.422002e+10,59.0,64504.0
9,1.422002e+10,59.0,64504.0


In [66]:
# --- Rebuild analysis APN_12 correctly from column 0 (floats) ---
analysis_apn12 = (
    final_analysis_df[0]
    .dropna()                 # IMPORTANT: do NOT turn blanks into 0
    .astype("Int64")          # removes the .0 cleanly
    .astype(str)
    .str.zfill(12)            # restores leading zeros like 014220...
)

# --- Our APN_12 (already correct, but normalize similarly) ---
our_apn12 = (
    levy_df["APN_12"]
    .astype(str)
    .str.replace(r"\D", "", regex=True)
    .str.zfill(12)
)

analysis_set = set(analysis_apn12)
our_set = set(our_apn12)

missing_in_ours = sorted(analysis_set - our_set)
extra_in_ours = sorted(our_set - analysis_set)

print("\n=== FINAL LEVY COMPARISON (CLEAN) ===")
print("Analysis unique APNs:", len(analysis_set))
print("Our unique APNs:", len(our_set))
print("Missing in ours:", len(missing_in_ours))
print("Extra in ours:", len(extra_in_ours))

if missing_in_ours:
    print("\nExamples missing in ours (first 25):")
    print(missing_in_ours[:25])

if extra_in_ours:
    print("\nExamples extra in ours (first 25):")
    print(extra_in_ours[:25])


KeyError: 'APN_12'

In [ ]:
missing_in_ours = sorted(analysis_set - our_set)
extra_in_ours = sorted(our_set - analysis_set)

print("\n=== FINAL LEVY COMPARISON ===")
print("Missing in ours:", len(missing_in_ours))
print("Extra in ours:", len(extra_in_ours))

if missing_in_ours:
    print("\nExamples missing in ours (first 25):")
    print(missing_in_ours[:25])

if extra_in_ours:
    print("\nExamples extra in ours (first 25):")
    print(extra_in_ours[:25])



=== FINAL LEVY COMPARISON ===
Missing in ours: 10984
Extra in ours: 10941

Examples missing in ours (first 25):
['000000000000', '142200050000', '142200060000', '142200140000', '142200150000', '142200160000', '142200170000', '142200180000', '142200190000', '142200200000', '142400080000', '142400090000', '142500010000', '142500020000', '142500030000', '142500040000', '221500030000', '221500040000', '221500050000', '222310010000', '222310020000', '222310030000', '222310050000', '222310060000', '222310070000']

Examples extra in ours (first 25):
['014220005000', '014220006000', '014220014000', '014220015000', '014220016000', '014220017000', '014220018000', '014220019000', '014220020000', '014240008000', '014240009000', '014250001000', '014250002000', '014250003000', '014250004000', '022150003000', '022150004000', '022150005000', '022231001000', '022231002000', '022231003000', '022231005000', '022231006000', '022231007000', '022231012000']


In [ ]:
# Focus only on missing APNs that ARE in mammoth_df
missing_in_mammoth_set = set(missing_in_mammoth)

# Normalize exemption APN sets
nontax_apns = set(
    nontax_df["APN_str"]
    .astype(str)
    .str.replace(r"\D","", regex=True)
    .str.zfill(12)
)

senior_apns = set(
    senior_df["APN_str"]
    .astype(str)
    .str.replace(r"\D","", regex=True)
    .str.zfill(12)
)

print("Of the 36 missing Mammoth APNs:")
print("In non-taxable list:", len(missing_in_mammoth_set & nontax_apns))
print("In senior list:", len(missing_in_mammoth_set & senior_apns))

print("\nExamples in non-taxable:")
print(list(missing_in_mammoth_set & nontax_apns)[:10])

print("\nExamples in senior:")
print(list(missing_in_mammoth_set & senior_apns)[:10])


Of the 36 missing Mammoth APNs:
In non-taxable list: 30
In senior list: 6

Examples in non-taxable:
['062200010100', '905001026000', '062200006100', '062040003100', '062200007000', '905001021000', '905001007000', '905001025000', '062200008100', '026120003000']

Examples in senior:
['035212014000', '035270009000', '040021007000', '062090011000', '039060011000', '060170023000']


### Export to CSV

In [68]:
# =========================
# EXPORT: Final 2025–26 Tax Levy CSV
# =========================

out_path = Path.home() / "Desktop" / "64504 Mammoth USD 01252026.csv"

levy_df.to_csv(
    out_path,
    index=False
)

print("Saved:", out_path)


Saved: /Users/ewilson/Desktop/64504 Mammoth USD 01252026.csv
